<a href="https://colab.research.google.com/github/Marssssss/OneAI/blob/main/OneAI_Workflow_StateGraph_CLI_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OneAI 确定性工作流编排 — 当前实现状态与 CLI 开放方案

> 2026-06-16
> 涵盖：WorkflowDag + StateGraph 完整实现分析 + CLI 开放方案设计

---

## 一、当前实现状态全览

### 1.1 WorkflowDag — DAG 无环工作流

**完整度：~70%**（定义/编译/验证/执行全链路存在，但部分环节有简化）

#### 数据结构

```rust
// config.rs — 用户定义层 (WorkflowConfig + StepConfig)
WorkflowConfig {
    name: String,
    steps: Vec<StepConfig>,        // 工作流步骤
    variables: HashMap<String, String>,  // 共享变量
    timeout_secs: Option<u64>,
    default_retry_policy: RetryPolicy,
    continue_on_failure: bool,
}

StepConfig {
    id: String,                     // 步骤ID
    depends_on: Vec<String>,        // 依赖的前置步骤
    tool: Option<String>,           // 调用的工具名
    tool_args: Option<Value>,       // 工具参数 (JSON)
    prompt: Option<String>,         // LLM 提示 (无工具时)
    requires_approval: bool,        // 是否需要人工审批
    timeout_secs: Option<u64>,      // 步骤级超时
    retry_policy: Option<RetryPolicy>, // 步骤级重试
}

// dag.rs — 编译后的 DAG (WorkflowDag + DagNode)
WorkflowDag {
    name: String,
    nodes: HashMap<String, DagNode>, // 步骤节点
    levels: Vec<Vec<String>>,         // 并行层级 (拓扑排序)
    roots: Vec<String>,               // 入口节点
    leaves: Vec<String>,              // 终端节点
}
```

#### 执行流程

```
WorkflowConfig (JSON/YAML)
  → compile(config) → WorkflowDag
  → validate(config, &dag) → ValidationResult (循环检测/孤立节点/依赖完整性)
  → WorkflowExecutor.execute(&dag, config) → WorkflowResult
      ├─ Level 0: 所有 root 步骤并行执行
      ├─ Level 1: 依赖 Level 0 的步骤并行执行
      ├─ Level N: ...
      ├─ 每步: 查找 Tool → 执行 → 超时+重试 → 审批门检查
      └─ 上下文: WorkflowContext 流经所有层级 (变量 + 步骤输出)
```

#### 关键能力

- ✅ JSON 定义 + 编译为 DAG
- ✅ 拓扑排序 → 自动并行层级
- ✅ 循环检测 + 独立节点检测 + 依赖完整性验证
- ✅ 逐步超时 + 重试策略 + 人工审批节点
- ✅ 上下文变量在步骤间传递
- ✅ AppSession.execute_workflow() 端到端调用

#### 缺失

- ⚠️ **步骤间变量模板插值未实现** — `{{variable}}` 语法在 StepConfig 中定义但 execute_step() 未做模板替换
- ⚠️ **无 LLM 推理步骤** — 步骤只能绑定 Tool 或纯 prompt；没有 NodeAction::LlmInfer 能力
- ⚠️ **失败策略单一** — continue_on_failure = true/false 二选一，无条件性跳过
- ⚠️ **无 YAML 格式** — 只有 JSON，无 YAML/TOML 解析

---

### 1.2 StateGraph — 有环状态图

**完整度：~35%**（定义完整、DFS 环检测存在，但**无执行器实现**）

#### 数据结构

```rust
// state_graph.rs — LangGraph 式有环图
NodeAction {
    LlmInfer { system_prompt_override, use_streaming },  // LLM 推理
    ToolCall { tool_name, args_template },                // 工具调用
    Delegate { agent_kind, task_template },               // 子代理委托
    HumanApproval { description },                        // 人工审批 (中断点)
    ConditionCheck { condition },                         // 条件路由
}

EdgeCondition {
    HasToolCalls,        // 模型输出有工具调用 → 路由到工具节点
    IsFinalAnswer,       // 模型输出是最终答案 → 路由到终端
    RequestsDelegation,  // 模型请求委托 → 路由到子代理
    ErrorOccurred,       // 发生错误 → 路由到恢复节点
    StateEquals { variable, value },  // 状态变量匹配
    Always,              // 无条件
    Custom { name, description },     // 自定义条件
}

StateGraph {
    name: String,
    nodes: HashMap<String, GraphNode>,
    edges: HashMap<String, Vec<GraphEdge>>,  // from → [edges with conditions]
    entry_point: String,
    terminal_nodes: Vec<String>,
}

GraphState {        // 执行时流经节点的状态
    conversation: Conversation,
    variables: HashMap<String, String>,
    last_result: Option<String>,
    last_error: Option<String>,
    should_terminate: bool,
}

GraphExecutionResult {
    final_state: GraphState,
    completed: bool,
    terminal_node: Option<String>,
    iterations: usize,
    interrupt_checkpoints: Vec<String>,
}
```

#### 关键设计

- ✅ 支持**有环边** — ReAct 循环 (Think→Act→Observe→Think) 的显式建模
- ✅ 支持**条件边** — 动态路由 (HasToolCalls/IsFinalAnswer/ErrorOccurred 等)
- ✅ 支持**中断点** — `interrupt: true` 标记节点，执行可暂停等待人工介入
- ✅ 支持**状态流** — `GraphState` 显式在节点间传递，非隐式累积
- ✅ DFS 环检测 (diagnostic, not validation — cycles are allowed)
- ✅ 5 种 NodeAction 涵盖 LLM/工具/委托/审批/条件
- ✅ 7 种 EdgeCondition 涵盖常见路由场景

#### 缺失（严重）

- ❌ **StateGraph 执行器未实现** — `GraphExecutionResult` 定义了但没有 `execute_state_graph()` 函数
- ❌ **NodeAction::LlmInfer 未接线** — 定义了 LLM 推理节点但无 Provider 依赖注入
- ❌ **NodeAction::Delegate 未接线** — 定义了子代理委托节点但无 SubAgentFactory
- ❌ **EdgeCondition 评估未实现** — 条件路由的判断逻辑没有实现
- ❌ **中断点恢复未实现** — `interrupt_checkpoints` 是空 Vec
- ❌ **args_template 插值未实现** — `{{variable}}` 模板语法未做变量替换

---

### 1.3 与 Agent Loop 的集成

**完整度：~10%**（基本未集成）

- `AgentDecision::SwitchParadigm` → 返回格式字符串，不触发 StateGraph
- `AgentDecision::Delegate` → 触发 `SubAgentWrapper::run()`, 不走 StateGraph
- WorkflowDag 执行独立于 AgentLoop — `AppSession.execute_workflow()` 是独立入口
- AgentLoop 内部没有 "触发预定义 Workflow" 的路径

---

### 1.4 CLI 开放状态

**完整度：0%**（CLI 中完全没有 Workflow/StateGraph 命令）

当前 CLI 支持的命令：

```
/help · /tools · /skills · /skill · /clear · /cost · /session · /paradigm · /domain · /compact · /tool · /new · /quit
```

没有任何 `/workflow` 或 `/wf` 相关命令。WorkflowDag 和 StateGraph 的能力完全锁在 Rust API 层，用户无法通过 CLI 交互使用。

---

## 二、CLI 开放方案设计

### 2.1 设计理念

OneAI 的确定性工作流编排应该以 **3 种方式开放**：

1. **命令式** — `/wf run <name>` 直接执行预定义工作流
2. **声明式** — `/wf define <json|yaml>` 即时定义并执行自定义工作流
3. **自动式** — Agent Loop 在特定决策路径自动触发预定义 StateGraph

这 3 种方式对应不同的使用场景：
- 命令式：开发者明确知道要做什么（如 "跑一次代码审查流程"）
- 声明式：开发者临时编排多步骤任务（如 "先搜索→再分析→最后写报告"）
- 自动式：Agent 智能判断当前任务适合某预定义流程（如 SwitchParadigm → 触发 ReAct StateGraph）

### 2.2 命令体系设计

```
/wf                        — 显示所有可用工作流
/wf list                   — 同上
/wf run <name>             — 执行预定义工作流
/wf run <name> --vars k=v  — 执行并传入变量
/wf define <json>          — 即时定义并执行 (JSON 格式)
/wf define --file <path>   — 从文件加载并执行
/wf show <name>            — 显示工作流 DAG 可视化 (ASCII)
/wf status                 — 显示当前运行中的工作流状态
/wf cancel                 — 取消当前运行中的工作流
/wf history                — 显示工作流执行历史
/wf graph <name>           — 显示 StateGraph 可视化 (ASCII)
```

### 2.3 预定义工作流注册

在 DomainPack 中声明预定义工作流：

```rust
// oneai-domain/src/coding_pack.rs — 新增工作流定义
pub fn coding_pack(project_dir: &str) -> MergedDomainPack {
    // ... 现有 5 层配置 ...

    // 第6层: 预定义工作流 (新增)
    let workflows = vec![
        code_review_workflow(),
        debug_workflow(),
        refactor_workflow(),
        test_workflow(),
    ];

    MergedDomainPack {
        // ... 现有配置 ...
        workflows,  // 新增字段
    }
}
```

#### 预定义工作流示例

**代码审查工作流 (DAG)**:

```json
{
  "name": "code-review",
  "description": "Systematic code review: read → analyze → report",
  "steps": [
    { "id": "read_changes", "tool": "shell", "tool_args": {"command": "git diff HEAD"}, "description": "Read recent changes" },
    { "id": "check_syntax", "tool": "shell", "tool_args": {"command": "cargo check"}, "depends_on": ["read_changes"], "description": "Check syntax correctness" },
    { "id": "run_tests", "tool": "shell", "tool_args": {"command": "cargo test"}, "depends_on": ["read_changes"], "description": "Run tests" },
    { "id": "analyze_quality", "prompt": "Analyze the code changes for quality issues...", "depends_on": ["read_changes"], "description": "LLM quality analysis" },
    { "id": "compile_report", "prompt": "Compile a review report from: syntax={{check_syntax_output}}, tests={{run_tests_output}}, quality={{analyze_quality_output}}", "depends_on": ["check_syntax", "run_tests", "analyze_quality"], "description": "Compile review report" }
  ]
}
```

**ReAct 循环 (StateGraph)**:

```json
{
  "name": "react-loop",
  "entry_point": "think",
  "nodes": {
    "think":   { "id": "think",   "action": { "LlmInfer": { "system_prompt_override": null, "use_streaming": true } } },
    "act":     { "id": "act",     "action": { "ToolCall": { "tool_name": "{{selected_tool}}", "args_template": "{{tool_args}}" } } },
    "observe": { "id": "observe", "action": { "ConditionCheck": { "condition": "has_more_tool_calls" } }, "interrupt": false },
    "end":     { "id": "end",     "action": { "LlmInfer": { "system_prompt_override": "Provide final answer", "use_streaming": false } } }
  },
  "edges": {
    "think":   [{ "from": "think", "to": "act", "condition": { "HasToolCalls": null } }, { "from": "think", "to": "end", "condition": { "IsFinalAnswer": null } }],
    "act":     [{ "from": "act", "to": "observe", "condition": { "Always": null } }],
    "observe": [{ "from": "observe", "to": "think", "condition": { "StateEquals": { "variable": "has_more_tool_calls", "value": "true" } } }, { "from": "observe", "to": "end", "condition": { "StateEquals": { "variable": "has_more_tool_calls", "value": "false" } } }]
  },
  "terminal_nodes": ["end"]
}
```

### 2.4 CLI 命令实现代码

#### handle_workflow_command() — 新增到 tui/mod.rs

```rust
fn handle_workflow_command(app: &mut App, session: &AppSession, parts: &[&str]) {
    if parts.len() < 2 {
        app.add_message(ChatRole::System,
            "Workflow commands:\n\
             /wf list              — Show available workflows\n\
             /wf run <name>        — Execute a workflow\n\
             /wf run <name> --vars — Execute with variables\n\
             /wf define <json>     — Define & execute from JSON\n\
             /wf define --file <p> — Define & execute from file\n\
             /wf show <name>       — Show DAG visualization\n\
             /wf status            — Show running workflow status\n\
             /wf cancel            — Cancel running workflow\n\
             /wf graph <name>      — Show StateGraph visualization");
        return;
    }

    match parts[1] {
        "list" => {
            // 从 DomainPack 获取预定义工作流
            let workflows = session.get_available_workflows();
            if workflows.is_empty() {
                app.add_message(ChatRole::System, "No predefined workflows available. Use /domain coding to load coding workflows.");
            } else {
                let mut lines = vec![format!("📋 Available Workflows ({})\n", workflows.len())];
                for wf in &workflows {
                    lines.push(format!("  • {} — {}", wf.name, wf.description));
                }
                lines.push("\nType /wf run <name> to execute".to_string());
                app.add_message(ChatRole::System, lines.join("\n"));
            }
        }

        "run" => {
            if parts.len() < 3 {
                app.add_message(ChatRole::Error, "Usage: /wf run <name> [--vars key=value ...]");
                return;
            }
            let wf_name = parts[2];

            // 解析 --vars 参数
            let mut vars = HashMap::new();
            for i in 3..parts.len() {
                if parts[i].starts_with("--vars") || parts[i] == "--vars" {
                    // 收集后续的 key=value 参数
                    for j in (i+1)..parts.len() {
                        if let Some((k, v)) = parts[j].split_once('=') {
                            vars.insert(k.to_string(), v.to_string());
                        }
                    }
                    break;
                }
            }

            // 获取工作流配置
            let config = session.get_workflow_config(wf_name);
            match config {
                Some(mut config) => {
                    // 合入用户变量
                    for (k, v) in vars {
                        config.variables.insert(k, v);
                    }

                    app.add_message(ChatRole::System,
                        format!("▶ Starting workflow '{}' with {} steps...", config.name, config.steps.len()));

                    // 执行工作流
                    let result = session.execute_workflow(&config).await;
                    match result {
                        Ok(result) => {
                            let mut lines = vec![format!("✅ Workflow '{}' completed", result.name)];
                            lines.push(format!("   Steps: {} total, {} completed, {} failed",
                                result.step_results.len(),
                                result.completed_steps().len(),
                                result.failed_steps().len()));
                            lines.push(format!("   Time: {}ms", result.total_time_ms));

                            // 显示每步结果摘要
                            for (step_id, step) in &result.step_results {
                                let status_icon = match step.status {
                                    StepStatus::Completed => "✅",
                                    StepStatus::Failed => "❌",
                                    StepStatus::Skipped => "⏭️",
                                    _ => "⏳",
                                };
                                lines.push(format!("   {} {}: {}", status_icon, step_id,
                                    step.output.map(|o| truncate(&o, 100)).unwrap_or_else(|| "no output".to_string())));
                            }

                            app.add_message(ChatRole::System, lines.join("\n"));
                        }
                        Err(e) => {
                            app.add_message(ChatRole::Error, format!("❌ Workflow failed: {}", e));
                        }
                    }
                }
                None => {
                    app.add_message(ChatRole::Error,
                        format!("Workflow '{}' not found. Use /wf list to see available workflows.", wf_name));
                }
            }
        }

        "define" => {
            // /wf define <json> 或 /wf define --file <path>
            if parts.len() < 3 {
                app.add_message(ChatRole::Error,
                    "Usage: /wf define <json_string> OR /wf define --file <path_to_json_or_yaml>");
                return;
            }

            let config = if parts[2] == "--file" {
                if parts.len() < 4 {
                    app.add_message(ChatRole::Error, "Usage: /wf define --file <path>");
                    return;
                }
                let path = parts[3];
                let content = std::fs::read_to_string(path).unwrap_or_else(|e| {
                    app.add_message(ChatRole::Error, format!("Cannot read file: {}", e));
                    String::new()
                });
                if content.is_empty() { return; }
                // 尝试 YAML → JSON 转换 (如果文件是 .yaml/.yml)
                if path.ends_with(".yaml") || path.ends_with(".yml") {
                    // yaml_to_json(content)  // 需要添加 serde_yaml 依赖
                    WorkflowConfig::from_json(&content) // 暂时只支持 JSON
                } else {
                    WorkflowConfig::from_json(&content)
                }
            } else {
                // 直接传入 JSON 字符串
                WorkflowConfig::from_json(parts[2])
            };

            match config {
                Ok(config) => {
                    app.add_message(ChatRole::System,
                        format!("▶ Executing custom workflow '{}' with {} steps...",
                            config.name, config.steps.len()));
                    let result = session.execute_workflow(&config).await;
                    // ... 同 run 的结果展示 ...
                }
                Err(e) => {
                    app.add_message(ChatRole::Error, format!("Invalid workflow definition: {}", e));
                }
            }
        }

        "show" => {
            // ASCII 可视化 DAG
            if parts.len() < 3 {
                app.add_message(ChatRole::Error, "Usage: /wf show <name>");
                return;
            }
            let config = session.get_workflow_config(parts[2]);
            match config {
                Some(config) => {
                    let dag = oneai_workflow::compile(&config);
                    let viz = render_dag_ascii(&dag);
                    app.add_message(ChatRole::System, format!("{}\n{}", config.name, viz));
                }
                None => {
                    app.add_message(ChatRole::Error, format!("Workflow '{}' not found.", parts[2]));
                }
            }
        }

        "graph" => {
            // ASCII 可视化 StateGraph
            if parts.len() < 3 {
                app.add_message(ChatRole::Error, "Usage: /wf graph <name>");
                return;
            }
            let graph = session.get_state_graph(parts[2]);
            match graph {
                Some(graph) => {
                    let viz = render_state_graph_ascii(&graph);
                    app.add_message(ChatRole::System, format!("{}\n{}", graph.name, viz));
                }
                None => {
                    app.add_message(ChatRole::Error, format!("StateGraph '{}' not found.", parts[2]));
                }
            }
        }

        _ => {
            app.add_message(ChatRole::Error, format!("Unknown workflow command: {}. Type /wf for help.", parts[1]));
        }
    }
}
```

#### DAG ASCII 可视化

```rust
fn render_dag_ascii(dag: &WorkflowDag) -> String {
    let mut lines = Vec::new();

    for (level_idx, level_ids) in &dag.levels {
        lines.push(format!("Level {}:", level_idx));
        for step_id in level_ids {
            let node = dag.get_node(step_id).unwrap();
            let tool_info = if let Some(tool) = &node.step.tool {
                format!(" → {}", tool)
            } else {
                " → LLM".to_string()
            };
            let deps = if node.depends_on.is_empty() {
                String::new()
            } else {
                format!(" (depends: {})", node.depends_on.join(", "))
            };
            lines.push(format!("  ├── {}{}{}", step_id, tool_info, deps));
        }
    }

    lines.join("\n")
}
```

输出示例：
```
Level 0:
  ├── read_changes → shell
Level 1:
  ├── check_syntax → shell (depends: read_changes)
  ├── run_tests → shell (depends: read_changes)
  ├── analyze_quality → LLM (depends: read_changes)
Level 2:
  ├── compile_report → LLM (depends: check_syntax, run_tests, analyze_quality)
```

#### StateGraph ASCII 可视化

```rust
fn render_state_graph_ascii(graph: &StateGraph) -> String {
    let mut lines = Vec::new();
    lines.push(format!("Entry: {} →", graph.entry_point));

    for (node_id, node) in &graph.nodes {
        let action_str = match &node.action {
            NodeAction::LlmInfer { .. } => "🧠 LLM",
            NodeAction::ToolCall { tool_name, .. } => format!("🔧 {}", tool_name),
            NodeAction::Delegate { agent_kind, .. } => format!("🤖 →{}", agent_kind),
            NodeAction::HumanApproval { .. } => "✋ Approval",
            NodeAction::ConditionCheck { condition } => format!("🔀 {}", condition),
        };
        let interrupt_str = if node.interrupt { " ⏸" } else { "" };
        lines.push(format!("  {} [{}]{}", node_id, action_str, interrupt_str));

        // 显示出边
        if let Some(edges) = graph.edges.get(node_id) {
            for edge in edges {
                let cond_str = match &edge.condition {
                    Some(EdgeCondition::HasToolCalls) => " [HasToolCalls]",
                    Some(EdgeCondition::IsFinalAnswer) => " [IsFinalAnswer]",
                    Some(EdgeCondition::Always) => "",
                    Some(c) => format!(" [{}]", condition_name(c)),
                    None => "",
                };
                lines.push(format!("    → {}{}", edge.to, cond_str));
            }
        }
    }

    lines.push(format!("Terminal: {}", graph.terminal_nodes.join(", ")));
    lines.join("\n")
}
```

输出示例：
```
Entry: think →
  think [🧠 LLM]
    → act [HasToolCalls]
    → end [IsFinalAnswer]
  act [🔧 {{selected_tool}}]
    → observe
  observe [🔀 has_more_tool_calls]
    → think [has_more_tool_calls=true]
    → end [has_more_tool_calls=false]
  end [🧠 LLM]
Terminal: end
```

---

## 三、StateGraph 执行器实现方案

这是当前最大的缺失——StateGraph 定义完整但无执行器。

### 3.1 执行器核心逻辑

```rust
// oneai-workflow/src/state_executor.rs (新增)

/// StateGraph 执行器 — 按图遍历节点，评估条件路由，处理中断点。
pub struct StateGraphExecutor {
    provider: Arc<dyn LlmProvider>,
    tools: Arc<tokio::sync::RwLock<HashMap<String, Arc<dyn Tool>>>>,
    sub_agent_factory: Arc<dyn SubAgentFactory>,
    approval_gate: Arc<dyn ApprovalGate>,
    max_iterations: usize,  // 防止无限循环
}

impl StateGraphExecutor {
    pub async fn execute(
        &self,
        graph: &StateGraph,
        initial_state: GraphState,
    ) -> Result<GraphExecutionResult> {
        let mut state = initial_state;
        let mut current_node_id = graph.entry_point.clone();
        let mut iterations = 0;
        let mut interrupt_checkpoints = Vec::new();

        while iterations < self.max_iterations {
            // 1. 获取当前节点
            let node = graph.get_node(&current_node_id)
                .ok_or_else(|| OneAIError::Workflow(
                    format!("Node '{}' not found in graph", current_node_id)
                ))?;

            // 2. 检查是否是终端节点
            if graph.terminal_nodes.contains(&current_node_id) {
                return Ok(GraphExecutionResult {
                    name: graph.name.clone(),
                    final_state: state,
                    completed: true,
                    terminal_node: Some(current_node_id),
                    iterations,
                    interrupt_checkpoints,
                });
            }

            // 3. 检查中断点
            if node.interrupt {
                // 保存检查点，暂停执行等待人工介入
                let checkpoint_id = format!("interrupt_{}_{}", graph.name, current_node_id);
                interrupt_checkpoints.push(checkpoint_id.clone());

                // 请求人工审批
                let approval = self.approval_gate.request_approval(
                    ApprovalRequest {
                        tool_name: "state_graph_interrupt".into(),
                        args: serde_json::json!({
                            "node": current_node_id,
                            "description": match &node.action {
                                NodeAction::HumanApproval { description } => description,
                                _ => "Interrupt point reached",
                            },
                            "state": state.variables,
                        }),
                        risk_level: RiskLevel::Medium,
                        permission_level: Some(PermissionLevel::Standard),
                        justification: "StateGraph interrupt point".into(),
                    }
                ).await?;

                match approval {
                    ApprovalResponse::Denied { reason } => {
                        state.should_terminate = true;
                        state.last_error = Some(format!("Interrupt denied: {}", reason));
                        return Ok(GraphExecutionResult {
                            name: graph.name.clone(),
                            final_state: state,
                            completed: false,
                            terminal_node: None,
                            iterations,
                            interrupt_checkpoints,
                        });
                    }
                    _ => { /* Continue execution */ }
                }
            }

            // 4. 执行节点动作
            let action_result = self.execute_node_action(&node.action, &mut state).await?;

            // 5. 更新状态
            state.last_result = Some(action_result.output.clone());
            state.last_error = action_result.error.clone();
            if state.should_terminate { break; }

            // 6. 评估出边条件 → 选择下一节点
            let edges = graph.get_edges_from(&current_node_id);
            let next_node = self.route_next_node(edges, &state)?;

            if next_node.is_none() {
                // 无匹配条件 → 默认终止
                break;
            }

            current_node_id = next_node.unwrap();
            iterations += 1;
        }

        Ok(GraphExecutionResult {
            name: graph.name.clone(),
            final_state: state,
            completed: state.should_terminate == false,
            terminal_node: None,
            iterations,
            interrupt_checkpoints,
        })
    }

    /// 执行节点动作
    async fn execute_node_action(
        &self,
        action: &NodeAction,
        state: &mut GraphState,
    ) -> Result<ActionResult> {
        match action {
            NodeAction::LlmInfer { system_prompt_override, use_streaming } => {
                // 构建推理请求
                let system_prompt = system_prompt_override
                    .clone()
                    .unwrap_or_default();
                let request = InferenceRequest {
                    conversation: state.conversation.clone(),
                    system_prompt,
                    max_tokens: None,
                    temperature: None,
                    thinking_budget: None,
                    tools: None,  // LLM 推理节点不需要工具定义
                };
                let response = self.provider.infer(&request).await?;
                // 更新对话状态
                state.conversation.messages.extend(response.content);
                Ok(ActionResult {
                    output: response.text_content(),
                    error: None,
                    decision: None,
                })
            }

            NodeAction::ToolCall { tool_name, args_template } => {
                // 模板插值: {{variable}} → state.variables[key]
                let resolved_name = interpolate_template(tool_name, &state.variables);
                let resolved_args = if let Some(template) = args_template {
                    let json_str = interpolate_template(template, &state.variables);
                    serde_json::from_str(&json_str)
                        .unwrap_or(serde_json::json!({}))
                } else {
                    serde_json::json!({})
                };

                let tools = self.tools.read().await;
                let tool = tools.get(&resolved_name)
                    .ok_or_else(|| OneAIError::Workflow(
                        format!("Tool '{}' not found", resolved_name)
                    ))?;

                let output = tool.execute(resolved_args).await?;
                // 将工具输出追加到对话
                state.conversation.messages.push(Message {
                    role: Role::ToolResult,
                    content: vec![ContentBlock::Text { text: output.content.clone() }],
                });
                Ok(ActionResult {
                    output: output.content,
                    error: output.error,
                    decision: None,
                })
            }

            NodeAction::Delegate { agent_kind, task_template } => {
                let task = interpolate_template(task_template, &state.variables);
                let kind = SubAgentKind::from_str(agent_kind)
                    .unwrap_or(SubAgentKind::Custom(agent_kind.clone()));
                let budget = TokenBudget::new(50000); // 默认子代理预算
                let sub_agent = self.sub_agent_factory.create(kind, budget)?;
                let result = sub_agent.run(&task).await?;
                // 将子代理摘要追加到对话
                state.conversation.messages.push(Message {
                    role: Role::Assistant,
                    content: vec![ContentBlock::Text { text: result.summary.clone() }],
                });
                Ok(ActionResult {
                    output: result.summary,
                    error: None,
                    decision: None,
                })
            }

            NodeAction::HumanApproval { description } => {
                // 已在主循环中处理 (interrupt check)
                Ok(ActionResult {
                    output: description.clone(),
                    error: None,
                    decision: None,
                })
            }

            NodeAction::ConditionCheck { condition } => {
                // 评估条件 → 设置状态变量
                let result = self.evaluate_condition(condition, state)?;
                state.variables.insert("_condition_result".to_string(), result.to_string());
                Ok(ActionResult {
                    output: result.to_string(),
                    error: None,
                    decision: None,
                })
            }
        }
    }

    /// 路由到下一个节点 — 评估所有出边条件
    fn route_next_node(
        &self,
        edges: Vec<&GraphEdge>,
        state: &GraphState,
    ) -> Result<Option<String>> {
        // 优先级: 条件边 > 无条件边
        for edge in edges {
            if let Some(condition) = &edge.condition {
                if self.evaluate_edge_condition(condition, state)? {
                    return Ok(Some(edge.to.clone()));
                }
            } else {
                // 无条件边 — 兜底路由
                return Ok(Some(edge.to.clone()));
            }
        }
        Ok(None)
    }

    /// 评估边条件
    fn evaluate_edge_condition(
        &self,
        condition: &EdgeCondition,
        state: &GraphState,
    ) -> Result<bool> {
        match condition {
            EdgeCondition::HasToolCalls => {
                // 检查 last_result 是否包含工具调用意图
                // (需要 OutputParser 解析)
                Ok(state.last_result.as_ref()
                    .map(|r| r.contains("tool_use") || r.contains("function_call"))
                    .unwrap_or(false))
            }
            EdgeCondition::IsFinalAnswer => {
                Ok(!self.evaluate_edge_condition(&EdgeCondition::HasToolCalls, state)?)
            }
            EdgeCondition::RequestsDelegation => {
                Ok(state.last_result.as_ref()
                    .map(|r| r.contains("delegate") || r.contains("sub_agent"))
                    .unwrap_or(false))
            }
            EdgeCondition::ErrorOccurred => {
                Ok(state.last_error.is_some())
            }
            EdgeCondition::StateEquals { variable, value } => {
                Ok(state.variables.get(variable) == Some(value))
            }
            EdgeCondition::Always => Ok(true),
            EdgeCondition::Custom { name, .. } => {
                // 自定义条件 — 需要注册评估函数
                // 当前无注册机制，默认 false
                tracing::warn!("Custom condition '{}' not registered, defaulting to false", name);
                Ok(false)
            }
        }
    }
}

/// 模板插值: {{variable}} → state.variables[key]
fn interpolate_template(template: &str, variables: &HashMap<String, String>) -> String {
    let mut result = template.to_string();
    for (key, value) in variables {
        result = result.replace(&format!("{{{{{}}}}}", key), value);
    }
    result
}
```

### 3.2 与 Agent Loop 集成

```rust
// agent_loop.rs — SwitchParadigm 决策触发 StateGraph

AgentDecision::SwitchParadigm { paradigm } => {
    // 查找 DomainPack 中是否有对应的 StateGraph 定义
    let graph_key = match paradigm {
        ParadigmKind::ReAct => "react-loop",
        ParadigmKind::Plan => "plan-workflow",
        ParadigmKind::Reflect => "reflect-workflow",
        ParadigmKind::Explore => "explore-workflow",
    };

    // 尝试获取预定义 StateGraph
    if let Some(graph) = self.domain_pack.get_state_graph(graph_key) {
        // 执行 StateGraph
        let executor = StateGraphExecutor::new(
            self.provider.clone(),
            self.tools.clone(),
            self.sub_agent_factory.clone(),
            self.approval_gate.clone(),
            50, // max iterations
        );
        let initial_state = GraphState::new();
        let result = executor.execute(&graph, initial_state).await?;
        // 将 StateGraph 输出注入到主 Agent Loop
        return Ok(result.final_state.last_result.unwrap_or_default());
    } else {
        // 无预定义 StateGraph → 语义切换 (当前实现)
        self.run_paradigm(paradigm, &state)?
    }
}
```

---

## 四、DomainPack 工作流声明

### 4.1 DomainPack 新增第6层

```rust
// domain_pack.rs — 新增 workflows 字段
pub struct DomainPack {
    // ... 现有 5 层 ...
    pub tools: Vec<Arc<dyn Tool>>,             // 第1层: 工具集
    pub context_sources: Vec<ContextSource>,    // 第2层: 上下文
    pub permission_profile: PermissionProfile,  // 第3层: 权限
    pub paradigm_strategies: ...,               // 第4层: 范式
    pub compression_template: ...,              // 第5层: 压缩

    // 新增第6层: 预定义工作流
    pub workflows: Vec<WorkflowConfig>,
    pub state_graphs: Vec<StateGraph>,
}
```

### 4.2 CodingPack 工作流定义

```rust
// coding_pack.rs — 预定义 4 个工作流
fn code_review_workflow() -> WorkflowConfig {
    WorkflowConfig::from_json(r#"{
        "name": "code-review",
        "description": "Systematic code review: diff → check → test → report",
        "steps": [
            { "id": "read_diff", "tool": "shell", "tool_args": {"command": "git diff HEAD"}, "description": "Read recent code changes" },
            { "id": "check_syntax", "tool": "shell", "tool_args": {"command": "cargo check 2>&1"}, "depends_on": ["read_diff"], "description": "Syntax check" },
            { "id": "run_tests", "tool": "shell", "tool_args": {"command": "cargo test 2>&1 | tail -20"}, "depends_on": ["read_diff"], "description": "Run tests" },
            { "id": "quality_review", "prompt": "Review the following code changes for correctness, style, and efficiency issues:\n{{read_diff_output}}", "depends_on": ["read_diff"], "description": "LLM quality review" },
            { "id": "compile_report", "prompt": "Compile a code review report:\n- Syntax: {{check_syntax_output}}\n- Tests: {{run_tests_output}}\n- Quality: {{quality_review_output}}", "depends_on": ["check_syntax", "run_tests", "quality_review"], "description": "Final report" }
        ]
    }"#).unwrap()
}

fn debug_workflow() -> WorkflowConfig {
    WorkflowConfig::from_json(r#"{
        "name": "debug",
        "description": "Systematic debugging: reproduce → isolate → fix → verify",
        "steps": [
            { "id": "reproduce", "tool": "shell", "tool_args": {"command": "{{reproduce_command}}"}, "description": "Reproduce the bug" },
            { "id": "search_code", "tool": "grep", "tool_args": {"pattern": "{{error_pattern}}", "path": "."}, "depends_on": ["reproduce"], "description": "Search for error patterns" },
            { "id": "analyze", "prompt": "Analyze this bug: error={{reproduce_output}}, related code={{search_code_output}}", "depends_on": ["search_code"], "description": "Root cause analysis" },
            { "id": "fix", "tool": "edit_file", "tool_args": {"path": "{{fix_file}}", "old_string": "{{fix_old}}", "new_string": "{{fix_new}}"}, "depends_on": ["analyze"], "description": "Apply fix", "requires_approval": true },
            { "id": "verify", "tool": "shell", "tool_args": {"command": "cargo test 2>&1 | tail -10"}, "depends_on": ["fix"], "description": "Verify fix" }
        ],
        "variables": {}
    }"#).unwrap()
}

fn refactor_workflow() -> WorkflowConfig {
    // 先分析 → 再规划 → 再执行 → 最后验证
    ...
}

fn test_workflow() -> WorkflowConfig {
    // 先理解代码 → 再生成测试 → 再运行 → 再修复失败测试
    ...
}
```

### 4.3 ReAct StateGraph 定义

```rust
// coding_pack.rs — 预定义 ReAct 循环 StateGraph
fn react_state_graph() -> StateGraph {
    let mut graph = StateGraph::new("react-loop", "think");

    // 节点
    graph.add_node(GraphNode {
        id: "think".into(),
        action: NodeAction::LlmInfer {
            system_prompt_override: None,
            use_streaming: true,
        },
        interrupt: false,
        metadata: HashMap::new(),
    });

    graph.add_node(GraphNode {
        id: "act".into(),
        action: NodeAction::ToolCall {
            tool_name: "{{selected_tool}}".into(),
            args_template: Some("{{tool_args}}".into()),
        },
        interrupt: false,
        metadata: HashMap::new(),
    });

    graph.add_node(GraphNode {
        id: "approve".into(),
        action: NodeAction::HumanApproval {
            description: "Approve tool execution before proceeding".into(),
        },
        interrupt: true,  // 中断点 — 需要人工审批
        metadata: HashMap::new(),
    });

    graph.add_node(GraphNode {
        id: "end".into(),
        action: NodeAction::LlmInfer {
            system_prompt_override: Some("Provide a final answer to the user's question.".into()),
            use_streaming: true,
        },
        interrupt: false,
        metadata: HashMap::new(),
    });

    // 边
    graph.add_edge(GraphEdge {
        from: "think".into(),
        to: "act".into(),
        condition: Some(EdgeCondition::HasToolCalls),
        metadata: HashMap::new(),
    });
    graph.add_edge(GraphEdge {
        from: "think".into(),
        to: "end".into(),
        condition: Some(EdgeCondition::IsFinalAnswer),
        metadata: HashMap::new(),
    });
    graph.add_edge(GraphEdge {
        from: "act".into(),
        to: "approve".into(),
        condition: Some(EdgeCondition::Always),
        metadata: HashMap::new(),
    });
    graph.add_edge(GraphEdge {
        from: "approve".into(),
        to: "think".into(),
        condition: Some(EdgeCondition::Always),
        metadata: HashMap::new(),
    });

    graph.add_terminal("end".into());

    graph
}
```

---

## 五、实现优先级路线图

### Phase 1: CLI 基础开放（1 周）

| 任务 | 文件 | 工作量 |
|------|------|--------|
| 新增 `/wf` 命令处理函数 | tui/mod.rs | 中 |
| 实现 `/wf list` — 列出预定义工作流 | tui/mod.rs | 低 |
| 实现 `/wf show` — DAG ASCII 可视化 | 新增 render 函数 | 低 |
| 实现 `/wf run` — 执行 WorkflowDag | tui/mod.rs + session.rs | 中 |
| 实现 `/wf define` — 即时定义执行 | tui/mod.rs | 中 |
| DomainPack 新增 workflows/state_graphs 字段 | domain_pack.rs | 低 |
| CodingPack 新增 4 预定义工作流 | coding_pack.rs | 中 |
| AppSession 新增 get_workflow_config() / get_available_workflows() | session.rs | 低 |

### Phase 2: StateGraph 执行器（2 周）

| 任务 | 文件 | 工作量 |
|------|------|--------|
| 实现 StateGraphExecutor | 新增 state_executor.rs | 高 |
| 实现 NodeAction 执行 (LlmInfer/ToolCall/Delegate) | state_executor.rs | 高 |
| 实现 EdgeCondition 评估 | state_executor.rs | 中 |
| 实现 {{variable}} 模板插值 | state_executor.rs | 低 |
| 实现中断点处理 + 检查点保存 | state_executor.rs | 中 |
| 实现 `/wf graph` — StateGraph ASCII 可视化 | render 函数 | 低 |
| 实现 `/wf graph run` — 执行 StateGraph | tui/mod.rs | 中 |
| 定义 react-loop StateGraph | coding_pack.rs | 中 |

### Phase 3: Agent Loop 集成（1 周）

| 任务 | 文件 | 工作量 |
|------|------|--------|
| SwitchParadigm → 触发预定义 StateGraph | agent_loop.rs | 中 |
| DomainPack.state_graphs 接入 | domain_pack.rs | 低 |
| 实现 `/wf status` — 运行状态追踪 | tui/mod.rs | 低 |
| 实现 `/wf cancel` — 取消运行 | tui/mod.rs | 低 |
| 实现 `/wf history` — 执行历史 | tui/mod.rs + persistence | 中 |

---

## 六、核心价值总结

OneAI 的确定性工作流编排 (WorkflowDag + StateGraph) 与所有竞品的根本差异在于：

| 竞品 | 编排方式 | 确定性 |
|------|---------|--------|
| Claude Code | LLM 是唯一编排器 | ❌ 概率性 |
| Codex CLI | LLM 是唯一编排器 | ❌ 概率性 |
| Aider | Architect/Editor 双模型 | ⚠️ 半确定性 (规划确定, 执行概率) |
| Cursor | LLM 是唯一编排器 | ❌ 概率性 |
| Devin | 多Agent协作, LLM编排 | ❌ 概率性 |
| **OneAI** | **WorkflowDag (确定性) + StateGraph (有环确定性) + LLM (概率性)** | **✅ 三模式可选** |

当开发者选择 `/wf run code-review`，他得到的是：
- **确定性**：5 个步骤按 DAG 拓扑顺序执行，每步绑定具体工具
- **可审计**：每步输出可追溯，审批节点显式标记
- **可中断**：在 HumanApproval 节点暂停等待人工介入
- **可复现**：同一工作流 + 同一输入 = 相同的步骤序列

这是所有竞品**做不到的**——它们的每一步都由 LLM 概率性决策，无法保证相同的步骤序列或中断点。

CLI 通过 `/wf` 命令开放这一能力后，OneAI 就真正兑现了"确定性工作流 + 概率性推理"双模式的核心差异化承诺。
